# Dataset split and text vectorization

Task #5 prepares the task 2 combined dataset for baseline classification. Use the Python environment from `requirements.txt`.

We preserve the original Toxic Chat test set and split eligible training groups 85/15, following the task 2 dataset contract and task 4 handoff. Identical normalized prompts stay together. TF-IDF is fitted on training text only.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'scripts/split_and_vectorize.py').is_file():
    ROOT = ROOT.parent
if not (ROOT / 'scripts/split_and_vectorize.py').is_file():
    raise RuntimeError('Open this notebook from the repository root or notebooks directory.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.split_and_vectorize import prepare_dataset


## Create the splits and features

This uses the untruncated `prompt` column and the existing `Safe`, `Toxic`, and `Jailbreak` labels. JBB behavior and judge records stay outside the classifier data. Rerunning this cell replaces the generated files in `data/processed/tfidf/`.


In [ ]:
output = ROOT / 'data/processed/tfidf'
report = prepare_dataset(
    ROOT / 'data/combined_risk_dataset.csv', output,
    validation_size=0.15, seed=42,
)
for name, split in report['splits'].items():
    print(f"{name}: {split['rows']} rows, {split['labels']}, shape={split['shape']}")
    print(f"  Rows with no TF-IDF features: {split['zero_feature_rows']}")
print('Training class weights:', report['class_weights'])


## Load the classifier inputs

Each label array matches the rows in its sparse matrix and CSV. The vectorizer is saved for inference. Use validation to compare models; reserve test features and labels for the final evaluation.


In [ ]:
import joblib
import numpy as np
from scipy.sparse import load_npz

X_train = load_npz(output / 'X_train.npz')
X_val = load_npz(output / 'X_val.npz')
y_train = np.load(output / 'y_train.npy', allow_pickle=False)
y_val = np.load(output / 'y_val.npy', allow_pickle=False)
vectorizer = joblib.load(output / 'tfidf_vectorizer.joblib')
class_weights = report['class_weights']
print('Train:', X_train.shape, y_train.shape)
print('Validation:', X_val.shape, y_val.shape)


The remaining files are `X_test.npz` and `y_test.npy`, the original split rows in CSV format, `class_weights.json`, and `split_report.json` with the input checksum and package versions.

See [the split guide](../docs/split-and-vectorize.md) for counts, limitations, and the task 6 handoff. No classifier is trained in this notebook.
